In [3]:
import os
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP

import pandas as pd
import psycopg2


def load_env_file(env_path: Path) -> None:
    """Load KEY=VALUE pairs from a local .env file into os.environ."""
    if not env_path.exists():
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


def get_connection():
    # Works when running `jupyter ... data/update_latest_price.ipynb` from repo root.
    project_root = Path.cwd()
    if not (project_root / "data" / "csv" / "price_history.csv").exists() and (project_root / "../data").exists():
        project_root = project_root.parent

    data_dir = project_root / "data"
    load_env_file(data_dir / ".env")

    return psycopg2.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        dbname=os.getenv("DB_NAME"),
    )


def quantize_decimal(value: Decimal | None, places: int) -> Decimal | None:
    if value is None:
        return None
    quant = Decimal("1").scaleb(-places)  # e.g. places=4 -> Decimal('0.0001')
    return value.quantize(quant, rounding=ROUND_HALF_UP)


# Toggle this to avoid accidental DB writes.
RUN_UPSERT = True

project_root = Path.cwd()
if not (project_root / "data" / "csv" / "price_history.csv").exists() and (project_root / "../data").exists():
    project_root = project_root.parent

data_dir = project_root / "data"

price_history_path = data_dir / "csv" / "price_history.csv"
product_path = data_dir / "csv" / "product.csv"

if not price_history_path.exists():
    raise FileNotFoundError(f"Missing CSV: {price_history_path}")
if not product_path.exists():
    raise FileNotFoundError(f"Missing CSV: {product_path}")

# Load as strings first, so we can compute with exact decimals.
price_history = pd.read_csv(price_history_path, sep=";", dtype=str, encoding="utf-8")
product = pd.read_csv(product_path, sep=";", dtype=str, encoding="utf-8")

required_ph_cols = {"prod_code", "price", "currency", "date", "id"}
missing_ph = required_ph_cols - set(price_history.columns)
if missing_ph:
    raise ValueError(f"price_history.csv missing columns: {sorted(missing_ph)}")

# Parse types
price_history["id"] = pd.to_numeric(price_history["id"], errors="coerce")
price_history["date_parsed"] = pd.to_datetime(
    price_history["date"], format="%d/%m/%y", errors="coerce"
)

# Convert numeric columns to Decimal

def to_decimal_or_none(x: object) -> Decimal | None:
    if x is None:
        return None
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s or s.lower() == "nan":
        return None
    return Decimal(s)


price_history["price_dec"] = price_history["price"].apply(to_decimal_or_none)

# We consider "latest" as: max(date_parsed), and if there are ties, max(id).
price_history = price_history.sort_values(["prod_code", "date_parsed", "id"]).copy()
price_history["prev_price_dec"] = price_history.groupby("prod_code")["price_dec"].shift(1)

latest_rows = price_history.groupby("prod_code", as_index=False).tail(1).copy()

# Compute latest_price fields

def compute_fields(row: pd.Series):
    prod_code = row["prod_code"]

    price = row["price_dec"]
    prev_price = row["prev_price_dec"]

    # last_updated_at derived from the latest observation date.
    last_dt = None
    if pd.notna(row["date_parsed"]):
        # Use midnight timestamp for stable comparisons.
        last_dt = row["date_parsed"].to_pydatetime()

    currency_code = row.get("currency")

    if price is None:
        return {
            "price": None,
            "currency_code": currency_code,
            "last_updated_at": last_dt,
            "value_change": None,
            "percentage_change": None,
            "change_direction": None,
        }

    if prev_price is None or prev_price == 0:
        value_change = None
        percentage_change = None
        change_direction = None
    else:
        value_change = price - prev_price
        percentage_change = (value_change / prev_price) * Decimal("100")
        if value_change > 0:
            change_direction = "increase"
        elif value_change < 0:
            change_direction = "decrease"
        else:
            change_direction = "flat"

    return {
        "price": quantize_decimal(price, 4),
        "currency_code": currency_code,
        "last_updated_at": last_dt,
        "value_change": quantize_decimal(value_change, 4),
        "percentage_change": quantize_decimal(percentage_change, 2),
        "change_direction": change_direction,
    }


computed = []
for _, r in latest_rows.iterrows():
    computed.append((r["prod_code"], compute_fields(r)))

latest_by_prod = {prod_code: fields for prod_code, fields in computed}

# Ensure every product exists in the DB upsert set (with NULLs if missing in history)
all_products = product[["prod_code"]].drop_duplicates().copy()

rows_to_upsert = []
for prod_code in all_products["prod_code"].tolist():
    fields = latest_by_prod.get(prod_code, {})
    rows_to_upsert.append(
        (
            prod_code,
            fields.get("price"),
            fields.get("currency_code"),
            fields.get("last_updated_at"),
            fields.get("value_change"),
            fields.get("percentage_change"),
            fields.get("change_direction"),
        )
    )

# Preview (first 10)
preview_df = pd.DataFrame(
    rows_to_upsert[:10],
    columns=[
        "prod_code",
        "price",
        "currency_code",
        "last_updated_at",
        "value_change",
        "percentage_change",
        "change_direction",
    ],
)
print(preview_df.to_string(index=False))

if RUN_UPSERT:
    conn = get_connection()
    conn.autocommit = False
    try:
        with conn.cursor() as cursor:
            insert_sql = """
                INSERT INTO price_latest (
                    prod_code,
                    price,
                    currency_code,
                    last_updated_at,
                    value_change,
                    percentage_change,
                    change_direction
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (prod_code)
                DO UPDATE SET
                    price = EXCLUDED.price,
                    currency_code = EXCLUDED.currency_code,
                    value_change = EXCLUDED.value_change,
                    percentage_change = EXCLUDED.percentage_change,
                    change_direction = EXCLUDED.change_direction,
                    last_updated_at = EXCLUDED.last_updated_at;
            """

            for (prod_code, price, currency_code, last_updated_at, value_change, percentage_change, change_direction) in rows_to_upsert:
                cursor.execute(
                    """
                        INSERT INTO price_latest (
                            prod_code, price, currency_code, last_updated_at,
                            value_change, percentage_change, change_direction
                        )
                        VALUES (%s, %s, %s, %s, %s, %s, %s)
                        ON CONFLICT (prod_code)
                        DO UPDATE SET
                            price = EXCLUDED.price,
                            currency_code = EXCLUDED.currency_code,
                            value_change = EXCLUDED.value_change,
                            percentage_change = EXCLUDED.percentage_change,
                            change_direction = EXCLUDED.change_direction,
                            last_updated_at = EXCLUDED.last_updated_at;
                    """,
                    (prod_code, price, currency_code, last_updated_at, value_change, percentage_change, change_direction),
                )

        conn.commit()
        print(f"Upserted price_latest rows: {len(rows_to_upsert)}")

        with conn.cursor() as cursor:
            cursor.execute("SELECT prod_code, price, currency_code, last_updated_at, value_change, percentage_change, change_direction FROM price_latest ORDER BY prod_code LIMIT 10;")
            rows = cursor.fetchall()

        check_df = pd.DataFrame(
            rows,
            columns=["prod_code", "price", "currency_code", "last_updated_at", "value_change", "percentage_change", "change_direction"],
        )
        print(check_df.to_string(index=False))
    except Exception as exc:
        conn.rollback()
        raise
    finally:
        conn.close()


 prod_code      price currency_code last_updated_at value_change percentage_change change_direction
   MOGAS95   132.0300           USD      2026-03-26      -3.5700             -2.63         decrease
   MOGAS92   127.4800           USD      2026-03-26      -3.5700             -2.72         decrease
    DO005S   215.5600           USD      2026-03-26      10.9400              5.35         increase
  FO180CST   662.8500           USD      2026-03-26      25.0200              3.92         increase
   MARINE5   806.1400           USD      2026-03-26      12.3300              1.55         increase
DATEDBRENT   117.4900           USD      2026-03-26       7.9100              7.22         increase
   BRENTKH   109.0500           USD      2026-03-26       7.7000              7.60         increase
       WTI    94.4800           USD      2026-03-26       2.1600              2.34         increase
    BACHHO   117.8900           USD      2026-03-26       4.8700              4.31         increase
